In [ ]:
%%time
%%capture
!pip install evaluate
!pip install datasets
!pip install "transformers==4.57.2"
!pip install sentencepiece
!pip install accelerate

In [ ]:
import os
import shutil
import numpy as np
import pandas as pd
import torch
import evaluate
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)
from transformers import DebertaV2Tokenizer, DebertaV2ForSequenceClassification

# Check device
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device count: {torch.cuda.device_count()}")

In [ ]:
%%time
MODEL_PATH = "./pretrain_mdeberta"

tokenizer = DebertaV2Tokenizer.from_pretrained(MODEL_PATH)
model = DebertaV2ForSequenceClassification.from_pretrained(MODEL_PATH, num_labels=2)

In [ ]:
%%capture
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
# model.to(device)
model.to(device)

In [ ]:
DATA_DIR = "./subtask1"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
DEV_DIR = os.path.join(DATA_DIR, "dev")
TEST_DIR = os.path.join(DATA_DIR, "test")

def load_split(split_dir):
    dfs = []
    if not os.path.exists(split_dir):
        print(f"Directory not found: {split_dir}")
        return pd.DataFrame()
        
    for file in os.listdir(split_dir):
        if file.endswith(".csv"):
            lang = file.replace(".csv", "")
            df = pd.read_csv(os.path.join(split_dir, file))
            df["lang"] = lang
            dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

# Load Datasets
print("Loading Train Data...")
raw_train_df = load_split(TRAIN_DIR)
print(f"Loaded {len(raw_train_df)} training examples")

print("Loading Dev Data (Used as internal Test)...")
raw_dev_df = load_split(DEV_DIR)
print(f"Loaded {len(raw_dev_df)} dev examples")

print("Loading Test Data (For Submission)...")
raw_test_df = load_split(TEST_DIR)
print(f"Loaded {len(raw_test_df)} test examples")

In [ ]:
# Pre-processing
if "polarization" in raw_train_df.columns:
    raw_train_df = raw_train_df.rename(columns={"polarization": "labels"})

if "polarization" in raw_dev_df.columns:
    raw_dev_df = raw_dev_df.rename(columns={"polarization": "labels"})

# Split Train into 95% Train / 5% Val
train_df, val_df = train_test_split(
    raw_train_df,
    test_size=0.05,
    stratify=raw_train_df["labels"],
    random_state=42,
    shuffle=True
)

# Use Dev as Internal Test
test_df = raw_dev_df.copy()

print("Shape after split:")
print(f"Train:      {train_df.shape}")
print(f"Validation: {val_df.shape}")
print(f"Test (Dev): {test_df.shape}")

In [ ]:
# Create Dataset Objects
train_dataset = Dataset.from_pandas(train_df[["text", "labels"]], preserve_index=False)
val_dataset = Dataset.from_pandas(val_df[["text", "labels"]], preserve_index=False)
test_dataset = Dataset.from_pandas(test_df[["text", "labels"]], preserve_index=False)

dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

dataset

In [ ]:
%%time
def tokenize_function(examples):
    return tokenizer(
        examples["text"], 
        padding="max_length", 
        truncation=True, 
        max_length=256
    )

encoded_dataset = dataset.map(tokenize_function, batched=True)
encoded_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
batch_size = 32
gradient_accumulation_steps = 2

In [ ]:
steps_per_epoch = len(dataset["train"]) // (batch_size * gradient_accumulation_steps)
# eval_steps = steps_per_epoch // 3
eval_steps = steps_per_epoch

print(f"Steps per epoch: {steps_per_epoch}")
print(f"Eval steps: {eval_steps}")

In [ ]:
training_args = TrainingArguments(
    output_dir="./output_results",
    num_train_epochs=30,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=1000,
    gradient_accumulation_steps=gradient_accumulation_steps,
    logging_steps=eval_steps,
    eval_steps=eval_steps,
    save_steps=eval_steps * 20,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    eval_strategy="steps",
    logging_dir="./logs",
    report_to="none",
    fp16=False,
)

metric = evaluate.load("f1")
metric_acc = evaluate.load("accuracy")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)
    f1 = metric.compute(predictions=predictions, references=labels, average="macro")["f1"]
    acc = metric_acc.compute(predictions=predictions, references=labels)["accuracy"]
    return {"f1": f1, "accuracy": acc}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

In [ ]:
%%time
trainer.train()

In [ ]:
# Evaluate on the "dev" dataset (which we treat as test)
print("Evaluating on Internal Test Set (Dev folder data)...")
preds_output = trainer.predict(encoded_dataset["test"])

pred_labels = np.argmax(preds_output.predictions, axis=1)
true_labels = preds_output.label_ids

print("\nClassification Report:")
print(classification_report(true_labels, pred_labels, target_names=["Not Polar (0)", "Polar (1)"], digits=4))

print(f"Macro F1: {f1_score(true_labels, pred_labels, average='macro'):.4f}")

In [ ]:
# Per-Language Analysis on Internal Test Set (Dev)
test_df["preds"] = pred_labels

print("\n=== Macro F1 per Language ===")
results = []
for lang in sorted(test_df["lang"].unique()):
    lang_df = test_df[test_df["lang"] == lang]
    f1 = f1_score(lang_df["labels"], lang_df["preds"], average="macro")
    acc = accuracy_score(lang_df["labels"], lang_df["preds"])
    print(f"{lang}: F1={f1:.4f}, Acc={acc:.4f}, Support={len(lang_df)}")
    results.append({"lang": lang, "f1_macro": f1, "accuracy": acc, "count": len(lang_df)})
    
# Optional: Display as DataFrame
results_df = pd.DataFrame(results)
print("\nAverage Macro F1 across languages:", results_df["f1_macro"].mean())

In [ ]:
SUBMISSION_DIR = "subtask_1"
if os.path.exists(SUBMISSION_DIR):
    shutil.rmtree(SUBMISSION_DIR)
os.makedirs(SUBMISSION_DIR)

print("Generating predictions for submission...")

# Create Dataset for submission files
submission_dataset = Dataset.from_pandas(raw_test_df[["text"]], preserve_index=False)
submission_tokenized = submission_dataset.map(tokenize_function, batched=True)
submission_tokenized.set_format(type="torch", columns=["input_ids", "attention_mask"])

# Predict
submission_preds_output = trainer.predict(submission_tokenized)
submission_labels = np.argmax(submission_preds_output.predictions, axis=1)

# Add predictions back to dataframe
raw_test_df["polarization"] = submission_labels

# Save individual files
languages = sorted(raw_test_df["lang"].unique())
print(f"Processing {len(languages)} languages for submission...")

for lang in languages:
    lang_df = raw_test_df[raw_test_df["lang"] == lang]
    output_df = lang_df[["id", "polarization"]]
    
    output_path = os.path.join(SUBMISSION_DIR, f"pred_{lang}.csv")
    output_df.to_csv(output_path, index=False)
    # print(f"Saved {output_path}")

print("Zipping prediction files...")
shutil.make_archive("subtask_1", "zip", SUBMISSION_DIR)
print("Created subtask_1.zip")